In [7]:
### Imports
import warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import os, time, logging


# Time series statistics
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
 
# Sklearn utilities
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             mean_absolute_error, mean_squared_error, r2_score,
                             confusion_matrix, classification_report)
 


 # Global configuration
# ---------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)
 
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
 
# Output directories
for d in ["data/raw", "data/processed", "models", "results/plots",
          "results/metrics"]:
    Path(d).mkdir(parents=True, exist_ok=True)
 
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (14, 5),
                     "font.size": 11})




In [8]:
# METHOD A: yfinance  (simplest, daily + hourly, free)
# -----------------------------------------------------------------------------
def extract_yfinance(ticker="BTC-USD",
                     start="2017-01-01",
                     end="2024-12-31",
                     intervals=("1d", "1h")):
    """
    Download OHLCV data from Yahoo Finance.
 
    Parameters
    ----------
    ticker    : str   — Yahoo Finance symbol, e.g. 'BTC-USD'
    start/end : str   — ISO date strings
    intervals : tuple — e.g. ('1d', '1h', '1wk')
 
    Returns
    -------
    dict[interval -> pd.DataFrame]
 
    Notes
    -----
    - yfinance returns a DatetimeIndex automatically.
    - Hourly data is limited to the last 730 days; adjust start accordingly.
    - Columns returned: Open, High, Low, Close, Adj Close, Volume
    """
    import yfinance as yf
 
    data = {}
    for iv in intervals:
        log.info("yfinance: downloading %s @ interval=%s", ticker, iv)
 
        # Hourly is capped at ~730 days by Yahoo
        if iv in ("1h", "90m", "30m"):
            df = yf.download(ticker, period="730d", interval=iv,
                             auto_adjust=True, progress=False)
        else:
            df = yf.download(ticker, start=start, end=end, interval=iv,
                             auto_adjust=True, progress=False)
 
        df.index = pd.to_datetime(df.index, utc=True).tz_localize(None)
        df.index.name = "datetime"
        df.columns = [c.lower() for c in df.columns]
        df.drop(columns=["adj close"], errors="ignore", inplace=True)
        df.sort_index(inplace=True)
        data[iv] = df
        log.info("  → %d rows, %s to %s", len(df),
                 df.index[0].date(), df.index[-1].date())
 
    return data